# 01 - Tiền xử lý & chọn đặc trưng MI

Luồng huấn luyện độc lập trên Colab: nhận `data/ton_iot.csv`, split trước, fit preprocessing/MI/scaler trên train-only, rồi lưu artifacts cho các notebook sau.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
os.makedirs(PROJECT_PATH, exist_ok=True)
%cd {PROJECT_PATH}
print('PROJECT_PATH =', PROJECT_PATH)

import time
from contextlib import contextmanager

@contextmanager
def step(name):
    t0 = time.time()
    print(f"\n[START] {name}", flush=True)
    try:
        yield
    finally:
        print(f"[DONE] {name} - {time.time() - t0:.1f}s", flush=True)


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip -q install pandas numpy scikit-learn pyarrow matplotlib seaborn


In [ ]:

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif

DATA_PATH = f'{PROJECT_PATH}/data/ton_iot.csv'
assert os.path.exists(DATA_PATH), f'Missing {DATA_PATH}'
OUT = f'{PROJECT_PATH}/data/colab_processed'
RESULTS = f'{PROJECT_PATH}/results'
MODELS = f'{PROJECT_PATH}/models'
os.makedirs(OUT, exist_ok=True); os.makedirs(RESULTS, exist_ok=True); os.makedirs(MODELS, exist_ok=True)
SEED = 42
TOP_K = 30

def make_oe():
    try: return OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    except TypeError: return OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

with step('Load CSV'):
    df = pd.read_csv(DATA_PATH)
print(f'[INFO] Raw shape: {df.shape}', flush=True)
label_col = 'label'
drop_cols = [label_col] + [c for c in ['type','attack_type','category'] if c in df.columns]
X_raw = df.drop(columns=drop_cols, errors='ignore')
y_raw = df[label_col]
le = LabelEncoder(); y = le.fit_transform(y_raw)

with step('Stratified train/test split'):
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=SEED)
print(f'[INFO] Train raw: {X_train_raw.shape}, Test raw: {X_test_raw.shape}', flush=True)
cat_cols = X_train_raw.select_dtypes(include=['object','category','bool']).columns.tolist()
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='mean'))]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('oe', make_oe())]), cat_cols),
], verbose_feature_names_out=False)

with step('Fit encoder/imputer on train and transform test'):
    X_train_all = np.nan_to_num(pre.fit_transform(X_train_raw).astype(np.float32))
    X_test_all = np.nan_to_num(pre.transform(X_test_raw).astype(np.float32))
print(f'[INFO] Encoded train: {X_train_all.shape}, Encoded test: {X_test_all.shape}', flush=True)
feature_names = list(pre.get_feature_names_out())
with step('Compute Mutual Information top-30'):
    mi = mutual_info_classif(X_train_all, y_train, random_state=SEED, discrete_features=False)
top_idx = np.argsort(mi)[::-1][:TOP_K]
top_features = [feature_names[i] for i in top_idx]

with step('Scale selected and all features'):
    scaler_top = MinMaxScaler(); X_train_top = scaler_top.fit_transform(X_train_all[:, top_idx]); X_test_top = scaler_top.transform(X_test_all[:, top_idx])
    scaler_all = MinMaxScaler(); X_train_all_scaled = scaler_all.fit_transform(X_train_all); X_test_all_scaled = scaler_all.transform(X_test_all)

with step('Save processed Parquet artifacts'):
    pd.DataFrame(X_train_top, columns=top_features).to_parquet(f'{OUT}/X_train_top30.parquet', index=False)
    pd.DataFrame(X_test_top, columns=top_features).to_parquet(f'{OUT}/X_test_top30.parquet', index=False)
    pd.DataFrame(X_train_all_scaled, columns=feature_names).to_parquet(f'{OUT}/X_train_all.parquet', index=False)
    pd.DataFrame(X_test_all_scaled, columns=feature_names).to_parquet(f'{OUT}/X_test_all.parquet', index=False)
    pd.DataFrame({'label': y_train}).to_parquet(f'{OUT}/y_train.parquet', index=False)
    pd.DataFrame({'label': y_test}).to_parquet(f'{OUT}/y_test.parquet', index=False)
mi_df = pd.DataFrame({'feature': feature_names, 'mi_score': mi}).sort_values('mi_score', ascending=False)
mi_df.to_csv(f'{RESULTS}/colab_mi_scores.csv', index=False)
metadata = {'protocol':'Colab independent training path from CSV; split first; fit preprocessing, MI top-30, and MinMaxScaler on train only.', 'seed':SEED, 'top_k':TOP_K, 'label_classes':[str(c) for c in le.classes_], 'top_features':top_features, 'n_train':int(len(y_train)), 'n_test':int(len(y_test)), 'n_all_features':int(X_train_all_scaled.shape[1])}
json.dump(metadata, open(f'{MODELS}/colab_preprocessing_metadata.json','w',encoding='utf-8'), indent=2)

plt.figure(figsize=(5,4)); pd.Series(y_raw).value_counts().sort_index().plot(kind='bar', color=['#4c78a8','#f58518']); plt.title('Label Distribution'); plt.xlabel('Label'); plt.ylabel('Count'); plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_colab_label_distribution.png', dpi=150); plt.show()
plt.figure(figsize=(9,8)); sns.barplot(data=mi_df.head(30), y='feature', x='mi_score', color='#4c78a8'); plt.title('Top-30 Mutual Information Features'); plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_colab_mi_top30.png', dpi=150); plt.show()
print('[OK] Saved Colab preprocessing artifacts to', OUT)
print('Train/Test:', X_train_top.shape, X_test_top.shape)
